## Leitura dos arquivos em PDF

## Bibliotecas

In [1]:
import os
import sys
import re

import pdfplumber
import spacy

## Funções e classes

In [2]:
sys.path.append(os.path.abspath(".."))

from src.utils import time_it

## Configurações

In [3]:
PATH_TO_FILES = '../data/'

PADROES = {
    # CPFs:
    r'\b\d{3}\.?\d{3}\.?\d{3}-?\d{2}\b': '[CPF]',
    # Telefones (diversos formatos):
    r'\(?\d{2,3}\)?\s?\d{4,5}-?\d{4}\b': '[TELEFONE]',
    # E-mails:
    r'\S+@\S+': '[EMAIL]',
    # Contas bancárias:
    r'\b\d{4,6}-?\d{1}\b': '[CONTA]',

    # Nomes próprios simples (capitalizados e isolados)
    r'\b[A-ZÁÉÍÓÚÂÊÔÃÕÇ][a-záéíóúâêôãõç]+(?: [A-ZÁÉÍÓÚÂÊÔÃÕÇ][a-záéíóúâêôãõç]+)*\b': '[NOME]'
}

## Importação dos arquivos

In [4]:
files = [f for f in os.listdir(PATH_TO_FILES) if '.pdf' in f]

### Código original

In [5]:
_file = files[2]

texto = ""
with pdfplumber.open(PATH_TO_FILES + _file) as pdf:
    for page in pdf.pages:
        texto += page.extract_text() or ""

print(texto)

Comprovante de Aplicação em Renda Fixa
Conta/DV Titular
003130320 MATHEUS ROSSO
Produto Índice Emissor CNPJ Emissor
CDB Pré-Fixado 15.00% a.a Banco BMG 61.186.680/0001-74
Data da Aplicação Data da Liquidação Data do Vencimento Liquidez
02/03/2023 02/03/2023 13/08/2026 Somente no Vencimento
Valor Aplicação Autenticação Eletrônica
R$ 1.000,00 5182122864D2378612FC3D03121D91E0416943E8
As notas de negociação estarão disponíveis para download em d+1 (a contar da data de liquidação) na área do Portal de Clientes
destinada aos relatórios.
Fique atento! Títulos emitidos pelo Banco Banco BMG contam com garantia do FGC somente até R$ 250 mil. Certifique-se de que o
somatório das diferentes emissões de um mesmo Banco, aqui ou em outros distribuidores, não ultrapasse tal valor. O Banco BTG
Pactual S.A. não garante títulos emitidos por outras instituições financeiras. Em caso de dúvidas, clique aqui ou leia nosso Termo de
Ciência.


### Função que implementa a importação + checagem

In [6]:
@time_it
def read_pdf(pdf_path: str, min_chars: int = 1, verbose: str = True) -> str | None:
    """
    ...
    """
    texto_total = ""
    textual_pages, image_pages = 0, 0

    # Inicia o núcleo da função:
    try:
        # Leitura do arquivo em PDF:
        with pdfplumber.open(pdf_path) as pdf:
            # Iteração sobre todas as páginas do arquivo:
            for page in pdf.pages:
                # Leitura do conteúdo textual da página:
                text = (page.extract_text() or "").strip()

                # Leitura do conteúdo de imagem da página:
                images = page.images # Lista de imagens na página.

                # Checando se há texto ou imagem na página:
                has_text = len(text) >= min_chars
                has_image = len(images) > 0

                if has_text:
                    textual_pages += 1 # Número de páginas com texto.
                    texto_total += text + "\n" # Conteúdo textual do arquivo.

                if has_image:
                    image_pages += 1 # Número de páginas com imagem.

        total_pages = len(pdf.pages) # Número total de páginas.

        # Classificação final:
        if (textual_pages > 0) & (image_pages == 0):
            if verbose:
                print('O PDF contém apenas textos.')
            return texto_total.strip()
        
        elif (textual_pages == 0) & (image_pages > 0):
            if verbose:
                print('O PDF é uma imagem (não contém texto selecionável).')
            return None

        elif (image_pages == 0) & (textual_pages == 0):
            if verbose:
                print('O PDF é um arquivo vazio.')
            return None

        else:
            if verbose:
                print('O PDF é híbrido (mistura de texto e imagem).')
            return texto_total.strip()

    except Exception as e:
        print(f"Erro ao processar o PDF: {e}")
        return None

_file = files[2]
_text_file = read_pdf(pdf_path=PATH_TO_FILES + _file, verbose=True)
print(_text_file)

O PDF é híbrido (mistura de texto e imagem).
Tempo de execução: 0.04 s

Comprovante de Aplicação em Renda Fixa
Conta/DV Titular
003130320 MATHEUS ROSSO
Produto Índice Emissor CNPJ Emissor
CDB Pré-Fixado 15.00% a.a Banco BMG 61.186.680/0001-74
Data da Aplicação Data da Liquidação Data do Vencimento Liquidez
02/03/2023 02/03/2023 13/08/2026 Somente no Vencimento
Valor Aplicação Autenticação Eletrônica
R$ 1.000,00 5182122864D2378612FC3D03121D91E0416943E8
As notas de negociação estarão disponíveis para download em d+1 (a contar da data de liquidação) na área do Portal de Clientes
destinada aos relatórios.
Fique atento! Títulos emitidos pelo Banco Banco BMG contam com garantia do FGC somente até R$ 250 mil. Certifique-se de que o
somatório das diferentes emissões de um mesmo Banco, aqui ou em outros distribuidores, não ultrapasse tal valor. O Banco BTG
Pactual S.A. não garante títulos emitidos por outras instituições financeiras. Em caso de dúvidas, clique aqui ou leia nosso Termo de
Ciênci

#### Testando a função

In [7]:
### DESENVOLVER LOGS!!!
textos = {}

for _file in files:
    textos[_file] = read_pdf(pdf_path=PATH_TO_FILES + _file, verbose=True)

Cannot set gray stroke color because /'P0' is an invalid float value
Cannot set gray stroke color because /'P1' is an invalid float value
Cannot set gray stroke color because /'P0' is an invalid float value
Cannot set gray stroke color because /'P1' is an invalid float value
Cannot set gray stroke color because /'P2' is an invalid float value
Cannot set gray stroke color because /'P3' is an invalid float value
Cannot set gray stroke color because /'P4' is an invalid float value


O PDF contém apenas textos.
Tempo de execução: 1.44 s

O PDF é híbrido (mistura de texto e imagem).
Tempo de execução: 0.12 s

O PDF é híbrido (mistura de texto e imagem).
Tempo de execução: 0.03 s

O PDF é híbrido (mistura de texto e imagem).
Tempo de execução: 0.05 s

O PDF é híbrido (mistura de texto e imagem).
Tempo de execução: 0.04 s

O PDF é híbrido (mistura de texto e imagem).
Tempo de execução: 0.03 s

O PDF é híbrido (mistura de texto e imagem).
Tempo de execução: 0.04 s

O PDF é híbrido (mistura de texto e imagem).
Tempo de execução: 0.03 s

O PDF é híbrido (mistura de texto e imagem).
Tempo de execução: 0.03 s

O PDF é híbrido (mistura de texto e imagem).
Tempo de execução: 0.04 s

O PDF é híbrido (mistura de texto e imagem).
Tempo de execução: 0.03 s

O PDF é híbrido (mistura de texto e imagem).
Tempo de execução: 0.03 s

O PDF é híbrido (mistura de texto e imagem).
Tempo de execução: 0.04 s

O PDF é híbrido (mistura de texto e imagem).
Tempo de execução: 0.03 s

O PDF é h

## Tratamento de textos

### Função para tratamento de dados textuais

In [10]:
def _remove_espacos(texto: str) -> str | None:
    """
    ...
    """
    if texto is not None:
        return re.sub(r'[ \t]+', ' ', texto).strip()
    else:
        return

def trata_textos(texto: str) -> str | None:
    """
    ...
    """
    # Mantendo apenas espaços simples ao longo de um texto:
    texto = _remove_espacos(texto)

    return texto

#### Testando a função

In [25]:
texto = textos[files[2]]
texto += '   teestandoo     '
print(texto)

Comprovante de Aplicação em Renda Fixa
Conta/DV Titular
003130320 MATHEUS ROSSO
Produto Índice Emissor CNPJ Emissor
CDB Pré-Fixado 15.00% a.a Banco BMG 61.186.680/0001-74
Data da Aplicação Data da Liquidação Data do Vencimento Liquidez
02/03/2023 02/03/2023 13/08/2026 Somente no Vencimento
Valor Aplicação Autenticação Eletrônica
R$ 1.000,00 5182122864D2378612FC3D03121D91E0416943E8
As notas de negociação estarão disponíveis para download em d+1 (a contar da data de liquidação) na área do Portal de Clientes
destinada aos relatórios.
Fique atento! Títulos emitidos pelo Banco Banco BMG contam com garantia do FGC somente até R$ 250 mil. Certifique-se de que o
somatório das diferentes emissões de um mesmo Banco, aqui ou em outros distribuidores, não ultrapasse tal valor. O Banco BTG
Pactual S.A. não garante títulos emitidos por outras instituições financeiras. Em caso de dúvidas, clique aqui ou leia nosso Termo de
Ciência.   teestandoo     


In [26]:
texto = trata_textos(texto=texto)
print(texto)

Comprovante de Aplicação em Renda Fixa
Conta/DV Titular
003130320 MATHEUS ROSSO
Produto Índice Emissor CNPJ Emissor
CDB Pré-Fixado 15.00% a.a Banco BMG 61.186.680/0001-74
Data da Aplicação Data da Liquidação Data do Vencimento Liquidez
02/03/2023 02/03/2023 13/08/2026 Somente no Vencimento
Valor Aplicação Autenticação Eletrônica
R$ 1.000,00 5182122864D2378612FC3D03121D91E0416943E8
As notas de negociação estarão disponíveis para download em d+1 (a contar da data de liquidação) na área do Portal de Clientes
destinada aos relatórios.
Fique atento! Títulos emitidos pelo Banco Banco BMG contam com garantia do FGC somente até R$ 250 mil. Certifique-se de que o
somatório das diferentes emissões de um mesmo Banco, aqui ou em outros distribuidores, não ultrapasse tal valor. O Banco BTG
Pactual S.A. não garante títulos emitidos por outras instituições financeiras. Em caso de dúvidas, clique aqui ou leia nosso Termo de
Ciência. teestandoo


In [ ]:
for _chave in textos.keys():
    textos[_chave] = trata_textos(textos[_chave])

## Anonimização dos dados

### Função para anonimização de padrões simples

In [19]:
PADROES

{'\\b\\d{3}\\.?\\d{3}\\.?\\d{3}-?\\d{2}\\b': '[CPF]',
 '\\(?\\d{2,3}\\)?\\s?\\d{4,5}-?\\d{4}\\b': '[TELEFONE]',
 '\\S+@\\S+': '[EMAIL]',
 '\\b\\d{4,6}-?\\d{1}\\b': '[CONTA]'}

In [28]:
def anonimizar_texto(texto: str, padroes: dict) -> str | None:
    """
    ...
    """
    if texto is not None:
        for padrao, substituto in padroes.items():
            texto = re.sub(padrao, substituto, texto)

        return texto
    else:
        return

#### Testando a função

In [20]:
texto = textos[files[2]]
texto += '030.300.290-50'
print(texto)

Comprovante de Aplicação em Renda Fixa
Conta/DV Titular
003130320 MATHEUS ROSSO
Produto Índice Emissor CNPJ Emissor
CDB Pré-Fixado 15.00% a.a Banco BMG 61.186.680/0001-74
Data da Aplicação Data da Liquidação Data do Vencimento Liquidez
02/03/2023 02/03/2023 13/08/2026 Somente no Vencimento
Valor Aplicação Autenticação Eletrônica
R$ 1.000,00 5182122864D2378612FC3D03121D91E0416943E8
As notas de negociação estarão disponíveis para download em d+1 (a contar da data de liquidação) na área do Portal de Clientes
destinada aos relatórios.
Fique atento! Títulos emitidos pelo Banco Banco BMG contam com garantia do FGC somente até R$ 250 mil. Certifique-se de que o
somatório das diferentes emissões de um mesmo Banco, aqui ou em outros distribuidores, não ultrapasse tal valor. O Banco BTG
Pactual S.A. não garante títulos emitidos por outras instituições financeiras. Em caso de dúvidas, clique aqui ou leia nosso Termo de
Ciência.030.300.290-50


In [24]:
texto = anonimizar_texto(texto=texto, padroes=PADROES)
print(texto)

Comprovante de Aplicação em Renda Fixa
Conta/DV Titular
003130320 MATHEUS ROSSO
Produto Índice Emissor CNPJ Emissor
CDB Pré-Fixado 15.00% a.a Banco BMG 61.186.680/0001-74
Data da Aplicação Data da Liquidação Data do Vencimento Liquidez
02/03/2023 02/03/2023 13/08/2026 Somente no Vencimento
Valor Aplicação Autenticação Eletrônica
R$ 1.000,00 5182122864D2378612FC3D03121D91E0416943E8
As notas de negociação estarão disponíveis para download em d+1 (a contar da data de liquidação) na área do Portal de Clientes
destinada aos relatórios.
Fique atento! Títulos emitidos pelo Banco Banco BMG contam com garantia do FGC somente até R$ 250 mil. Certifique-se de que o
somatório das diferentes emissões de um mesmo Banco, aqui ou em outros distribuidores, não ultrapasse tal valor. O Banco BTG
Pactual S.A. não garante títulos emitidos por outras instituições financeiras. Em caso de dúvidas, clique aqui ou leia nosso Termo de
Ciência.[CPF]


In [29]:
for _chave in textos.keys():
    textos[_chave] = anonimizar_texto(textos[_chave], padroes=PADROES)

In [ ]:
import re
import spacy
import pdfplumber

# carregar modelo de linguagem para detecção de nomes e entidades (português)
nlp = spacy.load("pt_core_news_lg")  # ou "pt_core_news_md"

def extrair_texto_pdf(caminho_pdf):
    texto = ""
    with pdfplumber.open(caminho_pdf) as pdf:
        for pagina in pdf.pages:
            texto += pagina.extract_text() + "\n"
    return texto

def anonimizar_texto(texto):
    # --- regex para CPFs, CNPJs, valores, contas, e-mails etc ---
    padroes_regex = {
        r'\b\d{3}\.?\d{3}\.?\d{3}-?\d{2}\b': '[CPF]',
        r'\b\d{2}\.?\d{3}\.?\d{3}/?\d{4}-?\d{2}\b': '[CNPJ]',
        r'R?\$ ?\d{1,3}(?:\.\d{3})*,\d{2}': '[VALOR]',
        r'\b\d{4,6}-?\d{1}\b': '[CONTA]',
        r'\S+@\S+': '[EMAIL]'
    }

    for padrao, substituto in padroes_regex.items():
        texto = re.sub(padrao, substituto, texto)

    # --- detecção de nomes e entidades com spaCy ---
    doc = nlp(texto)
    for ent in doc.ents:
        if ent.label_ in ['PER', 'ORG', 'LOC']:  # pessoa, organização, local
            texto = texto.replace(ent.text, f"[{ent.label_}]")

    return texto

# Exemplo de uso:
texto_pdf = extrair_texto_pdf("comprovante.pdf")
texto_anonimo = anonimizar_texto(texto_pdf)

print(texto_anonimo[:1000])  # mostra início do texto anonimizado